Generative Pre-trained Transformer

In [68]:
import tiktoken
import torch
import torch.nn as nn
import torch.nn.functional as F

In [69]:
data_path = "/home/bukunmi/ml-journey/datasets/shakespearesonnet.txt"

with open(data_path, "r", encoding="utf-8") as f:
    raw_text = f.read()

In [70]:
corpus = raw_text[:150].strip()
print(f"Sample Corpus Details:\n{"="*30}\n{corpus}\n{"="*30}")

Sample Corpus Details:
I

From fairest creatures we desire increase,
That thereby beauty’s rose might never die,
But as the riper should by time decease,
His tender heir mig


In [71]:
enc = tiktoken.encoding_for_model("gpt-4o")
token_ids = enc.encode(corpus)
vocab_size = enc.max_token_value + 1

In [72]:
data_tensor = torch.tensor(token_ids, dtype=torch.long)

Batching

In [73]:
block_size = len(data_tensor)

x_batch = data_tensor[:-1].unsqueeze(0)
y_batch = data_tensor[1:].unsqueeze(0)

Embeddings

In [74]:
embedding_dim = 64

Attention

In [75]:
class CausalAttention(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.q = nn.Linear(dim, dim, bias=False)
        self.k = nn.Linear(dim, dim, bias=False)
        self.v = nn.Linear(dim, dim, bias=False)
    
    def forward(self, x):
        B, T, C = x.shape
        Q = self.q(x)
        K = self.k(x)
        V = self.v(x)

        scores = (Q @ K.transpose(-2, -1)) * (C ** -0.5)
        mask = torch.tril(torch.ones(T, T, device=x.device)).view(1, T, T)

        scores = scores.masked_fill(mask == 0, float('-inf'))
        probs = F.softmax(scores, dim=-1)
        return probs @ V

Transformer

In [76]:
class TransformerBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.ln1 = nn.LayerNorm(dim)
        self.attn = CausalAttention(dim)
        self.ln2 = nn.LayerNorm(dim)
        self.ffn = nn.Sequential(
            nn.Linear(dim, 4 * dim),
            nn.ReLU(),
            nn.Linear(4 * dim, dim)
        )
    
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x

Language head

In [77]:
class TinyGPT(nn.Module):
    def __init__(self, vocab_size, dim, max_len):
        super().__init__()
        self.vocab_size = vocab_size
        self.token_emb = nn.Embedding(vocab_size, dim)
        self.pos_emb = nn.Embedding(max_len, dim)
        self.block = TransformerBlock(dim)
        self.ln_f = nn.LayerNorm(dim)
        self.lm_head = nn.Linear(dim, vocab_size)
    
    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok_vectors = self.token_emb(idx)
        pos_vectors = self.pos_emb(torch.arange(T, device=idx.device))
        x = tok_vectors + pos_vectors

        x = self.block(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, self.vocab_size), targets.reshape(-1))
        
        return logits, loss

Training

In [78]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TinyGPT(vocab_size, embedding_dim, block_size + 1).to(device)
x_batch, y_batch = x_batch.to(device), y_batch.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.005)

for step in range(151):
    logits, loss = model(x_batch, y_batch)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 25 == 0:
        print(f"Step {step:3d} | Cross-Entropy Loss: {loss.item():.5f}")

Step   0 | Cross-Entropy Loss: 12.37713
Step  25 | Cross-Entropy Loss: 0.09050
Step  50 | Cross-Entropy Loss: 0.00472
Step  75 | Cross-Entropy Loss: 0.00304
Step 100 | Cross-Entropy Loss: 0.00260
Step 125 | Cross-Entropy Loss: 0.00232
Step 150 | Cross-Entropy Loss: 0.00209


In [79]:
model.eval()
with torch.no_grad():
    seed_token = token_ids[0]
    generated_indices = [seed_token]
    max_new_tokens = len(data_tensor) - 1

    for _ in range(max_new_tokens):
        context = generated_indices[-block_size:]
        input_tensor = torch.tensor([context], dtype=torch.long, device=device)
        logits, _ = model(input_tensor)

        next_token = torch.argmax(logits[0, -1, :]).item()
        generated_indices.append(next_token)
    
    print("="*30, "\n", "Original Corpus", "\n", "="*30, "\n", corpus, "\n", "="*30, "\n")
    print("="*30, "\n", "Generated Corpus", "\n", "="*30, "\n", enc.decode(generated_indices), "\n", "="*30)

 Original Corpus 
 I

From fairest creatures we desire increase,
That thereby beauty’s rose might never die,
But as the riper should by time decease,
His tender heir mig 

 Generated Corpus 
 I

From fairest creatures we desire increase,
That thereby beauty’s rose might never die,
But as the riper should by time decease,
His tender heir mig 


Prompted next-token generation

In [80]:
def generate_text(model, prompt, max_new_tokens=20):
    model.eval()
    generated_indices = enc.encode(prompt)

    with torch.no_grad():
        for _ in range(max_new_tokens):
            context = generated_indices[-block_size:]
            input_tensor = torch.tensor([context], dtype=torch.long, device=device)
            logits, _ = model(input_tensor)
            next_token_logits = logits[0, -1, :]
            next_token = torch.argmax(next_token_logits).item()
            generated_indices.append(next_token)
    return enc.decode(generated_indices)


In [81]:
prompt = "I am a girl"
print(generate_text(model, prompt, max_new_tokens=20))

I am a girlst creatures we desire increase,
That thereby beauty’s rose might never die,
But as the riper
